# Kafka + Spark Structured Streaming

End-to-end demo: a `rate` source streams rows into a Kafka topic `streaming_events`, and a second query reads them back and prints to console.

While these streams run you can inspect everything in the **Kafka UI** at http://localhost:8085:
- **Topics** tab -> `streaming_events` -> **Messages** shows the live records and partitions.
- **Consumer Groups** tab -> `spark-streaming-demo` shows the running group and its lag.



In [ ]:
# Spark Session + Kafka connector
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("kafka-structured-streaming")
    .master("spark://bd-spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
print("Spark UI:", spark.sparkContext.uiWebUrl)

### 1. Producer — writeStream to Kafka

Spark itself is the producer: a `rate` source generates rows, and the Kafka sink publishes them to topic `streaming_events` (topic is auto-created; to force it, run `docker compose exec kafka kafka-topics --bootstrap-server kafka:9092 --create --if-not-exists --topic streaming_events --partitions 1 --replication-factor 1` first).



In [ ]:
KAFKA_BOOTSTRAP = "kafka:9092"
TOPIC = "streaming_events"

rate_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

producer = (
    rate_df.selectExpr(
        "CAST(timestamp AS STRING) AS key",
        "CAST(value AS STRING) AS value",
    )
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("topic", TOPIC)
    .option("checkpointLocation", "/tmp/kafka-checkpoint-producer")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Producer query running:", producer.name)

### 2. Consumer — readStream from Kafka



In [ ]:
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("kafka.group.id", "spark-streaming-demo")
    .load()
)

events = kafka_df.selectExpr(
    "CAST(key AS STRING) AS key",
    "CAST(value AS STRING) AS value",
    "topic", "partition", "offset", "timestamp",
)

consumer = (
    events.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/kafka-checkpoint-consumer")
    .trigger(processingTime="10 seconds")
    .start()
)
print("Consumer query running:", consumer.name)

### 3. Watch it all

- **Kafka UI** http://localhost:8085 — Topics / Messages / Consumer Groups / lag while the streams run.
- **Spark UI** (printed below) — open the **Streaming** tab to see both queries, their progress, input rates, and batch durations.


In [ ]:
print("spark.ui:", spark.sparkContext.uiWebUrl)
print("streams active:", [q.name for q in spark.streams.active])

spark.streams.awaitAnyTermination()

In [ ]:
# Stop cleanly when done
for q in spark.streams.active:
    q.stop()
print("stopped stream(s)")